In [1]:
%load_ext rpy2.ipython


In [2]:
%%R
install.packages("sqldf")
install.packages("readr")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’

trying URL 'https://cran.rstudio.com/src/contrib/gsubfn_0.7.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/proto_1.0.0.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/RSQLite_3.52.0.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/chron_2.3-62.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/sqldf_0.4-12.tar.gz'

The downloaded source packages are in
	‘/tmp/RtmpMIBnN2/downloaded_packages’
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/readr_2.2.0.tar.gz'
Content type 'application/x-gzip' length 295630 bytes (288 KB)
downloaded 288 KB


The downloaded source packages are in
	‘/tmp/RtmpMIBnN2/downloaded_packages’


In [3]:
%%R
library(readr)
library(sqldf)


Loading required package: gsubfn
Loading required package: proto
Loading required package: RSQLite
In addition: Warning message:
no DISPLAY variable so Tk is not available 


In [4]:
%%R

deliveries <- read_csv("deliveries.csv")
complaints <- read_csv("complaints.csv")
orders <- read_csv("orders.csv")
drivers <- read_csv("drivers.csv")
incidents <- read_csv("incidents.csv")
vehicles <- read_csv("vehicles.csv")

Rows: 950 Columns: 13
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (6): delivery_id, order_id, driver_id, vehicle_id, hub_id, delivery_status
dbl  (5): route_distance_km, manual_route_override_count, proof_of_completio...
dttm (2): dispatch_time, delivery_completed_at

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 320 Columns: 10
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (7): complaint_id, customer_id, order_id, complaint_type, channel, seve...
dbl  (2): resolution_days, compensation_amount
dttm (1): created_at

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 1250 Columns: 11
── Column specification ────────────────────────────────────────────────────

In [7]:
%%R

colnames(deliveries)

 [1] "delivery_id"                   "order_id"                     
 [3] "driver_id"                     "vehicle_id"                   
 [5] "hub_id"                        "dispatch_time"                
 [7] "delivery_completed_at"         "delivery_status"              
 [9] "route_distance_km"             "manual_route_override_count"  
[11] "proof_of_completion_missing"   "customer_rating_post_delivery"
[13] "fuel_or_charge_cost"          


In [8]:
%%R

sqldf("
SELECT delivery_status,
       COUNT(delivery_id) AS total_deliveries
FROM deliveries
GROUP BY delivery_status
ORDER BY total_deliveries DESC
")

  delivery_status total_deliveries
1          OnTime              616
2         Delayed              202
3          Failed              132


In [9]:
%%R

sqldf("
SELECT driver_id,
       SUM(manual_route_override_count) AS total_overrides
FROM deliveries
GROUP BY driver_id
ORDER BY total_overrides DESC
LIMIT 10
")

   driver_id total_overrides
1       D127              17
2       D130              16
3       D087              16
4       D131              15
5       D108              15
6       D105              14
7       D069              14
8       D028              13
9       D017              13
10      D104              12


In [10]:
%%R

sqldf("
SELECT hub_id,
       COUNT(delivery_id) AS total_deliveries,
       AVG(route_distance_km) AS avg_distance
FROM deliveries
GROUP BY hub_id
ORDER BY total_deliveries DESC
")

  hub_id total_deliveries avg_distance
1    H01              136     13.64331
2    H08              128     12.81547
3    H04              127     13.38457
4    H03              119     14.51555
5    H07              115     14.28696
6    H05              115     14.32165
7    H02              106     14.16915
8    H06              104     14.41221


In [11]:
%%R

sqldf("
SELECT delivery_status,
       AVG(customer_rating_post_delivery) AS avg_rating
FROM deliveries
GROUP BY delivery_status
")

  delivery_status avg_rating
1         Delayed   3.114975
2          Failed   3.049313
3          OnTime   4.283273


In [12]:
%%R

sqldf("
SELECT proof_of_completion_missing,
       COUNT(*) AS total_cases
FROM deliveries
GROUP BY proof_of_completion_missing
")

  proof_of_completion_missing total_cases
1                           0         881
2                           1          69
